# تحلیل توزیع Supercell + مدل IFC کامل — دیتاست کامل MAX Phase (بدون فیلتر اتم)

## هدف این Notebook
1. **بخش A (تحلیلی):** بررسی توزیع `n_atoms` و `n_supercell` روی **تمام ۳۵۸ ماده** (بدون فیلتر `n_atoms ≤ 12` که در نسخه‌های قبلی اعمال می‌شد) تا سقف درست (`MAX_SUPERCELL`) برای طراحی مدل تعیین شود.
2. **بخش B (مدل):** بعد از دیدن نتیجه‌ی بخش A، مدل Dual Graph GNN بازنویسی می‌شود تا کل ماتریس IFC `(n_atoms, n_supercell, 3, 3)` را پیش‌بینی کند، نه فقط بلوک سلول مرجع.

⚠️ **این Notebook را در دو مرحله اجرا کنید:**
- ابتدا فقط سلول‌های بخش A را اجرا کنید و خروجی (آمار توزیع) را برای ادامه‌ی کار ارسال کنید.
- بخش B بعداً بر اساس نتایج بخش A تکمیل خواهد شد.


In [ ]:
!pip install -q phonopy -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("✅ نصب شد")

In [ ]:
import os
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

from tqdm.notebook import tqdm

from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

print("✅ ایمپورت‌ها آماده شدند")

## مسیرهای داده

In [ ]:
FC_DIR     = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
POSCAR_DIR = '/kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar'
BANDS_DIR  = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'

for name, path in [('FC_DIR', FC_DIR), ('POSCAR_DIR', POSCAR_DIR), ('BANDS_DIR', BANDS_DIR)]:
    exists = '✅' if os.path.exists(path) else '❌ یافت نشد'
    print(f"{exists}  {name}  →  {path}")

## بخش A: تحلیل توزیع n_atoms و n_supercell (بدون هیچ فیلتری)

این سل فقط شکل فایل‌های `.fc` را می‌خواند (`n_unitcell` و `n_supercell`) — **بدون** اجرای
الگوریتم کشف خودکار supercell (که کندتر است) — تا سریعاً توزیع کلی روی همه‌ی ۳۵۸ ماده به‌دست آید.


In [ ]:
FC_DIR_PATH = Path(FC_DIR)
fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
print(f"تعداد کل فایل‌های .fc موجود: {len(fc_files)}")

shape_records = []
failed = []

for formula, fc_path in tqdm(fc_files.items(), desc="خواندن شکل فایل‌های FC"):
    try:
        with open(fc_path, 'r') as f:
            first_line = f.readline().strip().split()
        n_unitcell = int(first_line[0])
        n_supercell = int(first_line[1])
        n_images = n_supercell // n_unitcell if n_unitcell > 0 else 0
        shape_records.append({
            'formula': formula,
            'n_atoms': n_unitcell,
            'n_supercell': n_supercell,
            'n_images': n_images,
        })
    except Exception as e:
        failed.append((formula, str(e)))

df_shapes = pd.DataFrame(shape_records)
print(f"\n✅ خوانده‌شده: {len(df_shapes)}  |  ❌ خطا: {len(failed)}")
if failed[:5]:
    print("نمونه خطاها:", failed[:5])

In [ ]:
print("="*70)
print("📊 آمار توصیفی n_atoms (تعداد اتم در سلول واحد)")
print("="*70)
print(df_shapes['n_atoms'].describe())
print()
print("توزیع فراوانی n_atoms:")
print(df_shapes['n_atoms'].value_counts().sort_index())

print("\n" + "="*70)
print("📊 آمار توصیفی n_supercell (تعداد اتم در سوپرسل کامل)")
print("="*70)
print(df_shapes['n_supercell'].describe())
print()
print("توزیع فراوانی n_supercell:")
print(df_shapes['n_supercell'].value_counts().sort_index())

print("\n" + "="*70)
print("📊 آمار توصیفی n_images (n_supercell / n_atoms)")
print("="*70)
print(df_shapes['n_images'].describe())
print()
print("توزیع فراوانی n_images:")
print(df_shapes['n_images'].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df_shapes['n_atoms'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_xlabel('n_atoms (سلول واحد)')
axes[0].set_ylabel('تعداد مواد')
axes[0].set_title('توزیع n_atoms')

axes[1].hist(df_shapes['n_supercell'], bins=30, color='darkorange', edgecolor='black')
axes[1].set_xlabel('n_supercell (کل سوپرسل)')
axes[1].set_title('توزیع n_supercell')

axes[2].hist(df_shapes['n_images'], bins=30, color='seagreen', edgecolor='black')
axes[2].set_xlabel('n_images (n_supercell / n_atoms)')
axes[2].set_title('توزیع n_images')

plt.tight_layout()
plt.savefig('/kaggle/working/supercell_distribution.png', dpi=150)
plt.show()

print(f"\n🎯 حداکثر n_supercell در کل دیتاست: {df_shapes['n_supercell'].max()}")
print(f"🎯 صدک ۹۵ام n_supercell: {df_shapes['n_supercell'].quantile(0.95):.0f}")
print(f"🎯 صدک ۹۹ام n_supercell: {df_shapes['n_supercell'].quantile(0.99):.0f}")
print(f"🎯 میانه‌ی n_supercell: {df_shapes['n_supercell'].median():.0f}")

In [ ]:
df_shapes.to_csv('/kaggle/working/supercell_shape_analysis.csv', index=False)
print("💾 ذخیره شد: supercell_shape_analysis.csv")
print("\n⚠️ نتیجه‌ی این تحلیل (حداکثر/صدک ۹۵/۹۹ n_supercell) را برای ادامه‌ی ساخت مدل ارسال کنید.")
print("   بر اساس این عدد، MAX_SUPERCELL برای ابعاد خروجی مدل تعیین خواهد شد.")

---

## بخش B: مدل IFC کامل (placeholder — بعد از دیدن نتایج بخش A تکمیل می‌شود)

⏸️ این بخش هنوز کامل نشده است. لطفاً خروجی بخش A (به‌خصوص `n_supercell.max()`, `quantile(0.95)`, `quantile(0.99)`) را ارسال کنید تا `MAX_SUPERCELL` به‌صورت دقیق و مبتنی بر داده‌ی واقعی تعیین شود و کد آموزش مدل تکمیل گردد.
